<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تمت الترجمة بواسطة [سيرجي أوريشكوف](https://www.linkedin.com/in/sergeoreshkov/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> المهمة رقم 8 (تجريبي). الحل
## <center> تنفيذ التراجع عبر الإنترنت
    
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a8-demo-implementing-online-regressor) + [الحل](https://www.kaggle.com/kashnitsky/a8-demo-implementing-online-regressor-solution).**



سنقوم هنا بتنفيذ تراجع تم تدريبه باستخدام نزول التدرج العشوائي (SGD). املأ الرمز المفقود. إذا قمت بكل شيء بشكل صحيح، فسوف تجتاز اختبارًا بسيطًا مضمنًا.



## <center> الانحدار الخطي والهبوط التدرج العشوائي


In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.metrics import log_loss, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

%matplotlib inline
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler

قم بتنفيذ الفصل `SGDRegressor`. المواصفات:
- الفئة موروثة من `sklearn.base.BaseEstimator`
- يأخذ المنشئ المعلمات `eta` - خطوة التدرج ($10^{-3}$ بشكل افتراضي) و`n_epochs` - عدد تمريرات مجموعة البيانات (3 بشكل افتراضي)
- يقوم المُنشئ أيضًا بإنشاء قوائم `mse_` و`weights_` لتتبع متوسط الخطأ التربيعي ومتجه الوزن أثناء تكرارات نزول التدرج
- يحتوي الفصل على أساليب `fit` و`predict`
- تأخذ الطريقة `fit` المصفوفة `X` والمتجه `y` (`numpy.array` كائنات) كمعلمات، وتُلحق عمودًا من الواحدات بـ `X` على الجانب الأيسر، وتبدأ متجه الوزن `w` بـ **أصفار** ثم تقوم بإنشاء `n_epochs` تكرارات تحديثات الوزن (يمكنك الرجوع إلى هذه [المقالة](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-8-vowpal-wabbit-fast-learning-with-gigabytes-of-data-60f750086237) للحصول على التفاصيل)، ولكل سجلات تكرار يعني الخطأ التربيعي ومتجه الوزن `w` في القوائم المقابلة التي أنشأناها في المُنشئ. 
- بالإضافة إلى ذلك، ستقوم الطريقة `fit` بإنشاء متغير `w_` لتخزين الأوزان التي تنتج الحد الأدنى من متوسط الأخطاء المربعة
- يقوم الأسلوب `fit` بإرجاع المثيل الحالي للفئة `SGDRegressor`، أي `self`
- تأخذ طريقة `predict` مصفوفة `X`، وتضيف عمودًا من الآحاد إلى الجانب الأيسر وتعيد متجه التنبؤ، باستخدام ناقل الوزن `w_`، الذي تم إنشاؤه بواسطة طريقة `fit`.


In [ ]:
class SGDRegressor(BaseEstimator):
    def __init__(self, eta=1e-3, n_epochs=3):
        self.eta = eta
        self.n_epochs = n_epochs
        self.mse_ = []
        self.weights_ = []

    def fit(self, X, y):
        X = np.hstack([np.ones([X.shape[0], 1]), X])

        w = np.zeros(X.shape[1])

        for it in tqdm(range(self.n_epochs)):
            for i in range(X.shape[0]):

                new_w = w.copy()
                new_w[0] += self.eta * (y[i] - w.dot(X[i, :]))
                for j in range(1, X.shape[1]):
                    new_w[j] += self.eta * (y[i] - w.dot(X[i, :])) * X[i, j]
                w = new_w.copy()

                self.weights_.append(w)
                self.mse_.append(mean_squared_error(y, X.dot(w)))

        self.w_ = self.weights_[np.argmin(self.mse_)]

        return self

    def predict(self, X):
        X = np.hstack([np.ones([X.shape[0], 1]), X])

        return X.dot(self.w_)


دعونا نختبر الخوارزمية على بيانات الطول/الوزن. سنتوقع الارتفاعات (بالبوصة) بناءً على الأوزان (بالرطل).


In [ ]:
data_demo = pd.read_csv("../../data/weights_heights.csv")

In [ ]:
plt.scatter(data_demo["Weight"], data_demo["Height"])
plt.xlabel("Weight (lbs)")
plt.ylabel("Height (Inch)")
plt.grid();

In [ ]:
X, y = data_demo["Weight"].values, data_demo["Height"].values


إجراء تقسيم التدريب/الاختبار وقياس البيانات.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=17
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape([-1, 1]))
X_valid_scaled = scaler.transform(X_valid.reshape([-1, 1]))


تم إنشاء القطار `SGDRegressor` باستخدام بيانات `(X_train_scaled, y_train)`. اترك قيم المعلمات الافتراضية في الوقت الحالي.


In [ ]:
# you code here
sgd_reg = SGDRegressor()
sgd_reg.fit(X_train_scaled, y_train)


ارسم مخططًا يتضمن عملية التدريب - اعتماد متوسط الخطأ التربيعي من رقم التكرار i-th SGD.

In [ ]:
# you code here
plt.plot(range(len(sgd_reg.mse_)), sgd_reg.mse_)
plt.xlabel("#updates")
plt.ylabel("MSE");


اطبع القيمة الدنيا لمتوسط الخطأ التربيعي وأفضل ناقل الأوزان.


In [ ]:
# you code here
np.min(sgd_reg.mse_), sgd_reg.w_


ارسم مخططًا لسلوك أوزان النماذج ($w_0$ و$w_1$) أثناء التدريب.


In [ ]:
# you code here
plt.subplot(121)
plt.plot(range(len(sgd_reg.weights_)), [w[0] for w in sgd_reg.weights_])
plt.subplot(122)
plt.plot(range(len(sgd_reg.weights_)), [w[1] for w in sgd_reg.weights_]);


قم بالتنبؤ بمجموعة الإيقاف `(X_valid_scaled, y_valid)` وتحقق من قيمة MSE.


In [ ]:
# you code here
sgd_holdout_mse = mean_squared_error(y_valid, sgd_reg.predict(X_valid_scaled))
sgd_holdout_mse


افعل نفس الشيء مع `LinearRegression` للفئة من `sklearn.linear_model`. تقييم MSE لمجموعة الإيقاف.

In [ ]:
# you code here
from sklearn.linear_model import LinearRegression

lm = LinearRegression().fit(X_train_scaled, y_train)
print(lm.coef_, lm.intercept_)
linreg_holdout_mse = mean_squared_error(y_valid, lm.predict(X_valid_scaled))
linreg_holdout_mse

In [ ]:
try:
    assert (sgd_holdout_mse - linreg_holdout_mse) < 1e-4
    print("Correct!")
except AssertionError:
    print(
        "Something's not good.\n Linreg's holdout MSE: {}"
        "\n SGD's holdout MSE: {}".format(linreg_holdout_mse, sgd_holdout_mse)
    )